In [105]:
import math
import random

In [106]:
# NNs are large mathematical expressions.
# We create data structure that maintains this expression

class Value:

    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0     # This is derivative of final output w.r.t self (this object)

        # This func is empty for those who does not formed by performing operation i.e leaf node
        self._backward = lambda: None

        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f'Value(data={self.data})'
    
    # operation methods
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        # It will calculate grad for each child of current object using chain rule
        # Means grad of child1 w.r.t L & grad of child2 w.r.t L
        def _backward():
            # for f(x,y) = x + y , df/dx = 1 & df/dy = 1
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        
        out._backward = _backward
        return out
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            # for f(x,y) = x * y, df/dx = y & df/dy = x
            self.grad += other.data * out.grad
            other.grad += self.data  * out.grad
        
        out._backward = _backward
        return out
    
    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only support int/float values as power"

        out = Value(self.data ** other, (self,), f'**{other}')

        def _backward():
            self.grad += (other * (self.data**(other - 1))) * out.grad
        out._backward = _backward
        return out
    
    # a/b = a * b**-1.
    def __truediv__(self, other):
        return self * other**-1
    
    def __neg__(self):
        return self * -1
    
    def __sub__(self, other):
        return self + (-other)
    
    # fallback function for __add__() & __mul__()
    def __radd__(self, other):
        return self + other
    def __rmul__(self, other):
        return self * other
    
    # tanh operation
    def tanh(self):
        x = self.data
        t = (math.exp(2 * x) - 1) / (math.exp(2 * x) + 1)
        out = Value(t, (self,), 'tanh')

        def _backward():
            # derivative of tanh(x) is 1 - tanh(x)^2
            self.grad += (1 - (t**2)) * out.grad
        
        out._backward = _backward
        return out
    
    # exponent operation
    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self,), 'exp')

        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        return out

    
    
    def backward(self):

        # It sets all instances in sequential order which resulted in self. Meaning, it finds all its childs nodes and sets them in sequential manner i.e topological graph (graph that only flows from left  to right)
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        
        # build topological graph
        build_topo(self)

        # set grad of L w.r.t itself (l) to 1. because initially, we set to 0.0
        self.grad = 1
        
        # calculate grads
        for node in reversed(topo):
            node._backward()
        


In [107]:
class Neuron:

    def __init__(self,nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))
    
    def __call__(self,x):
        # calculate weighted sum
        act = sum((wi * xi for wi,xi in zip(self.w, x)), self.b)
        # apply activation func
        out = act.tanh()
        return out
    
    def parameters(self):
        return self.w + [self.b]

class Layer:

    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]
    
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    
    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

class MLP:

    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

n = MLP(3,[4,4,1])




In [108]:
# input data
xs = [
    [1.0, -2.0, 4.0],
    [5.0, 9.0, -2.0],
    [10.0, 8.0, 3.0],
    [11.0, -12.0, 9.0],
]

ys = [1.0, -1.0, -1.0, 1.0]

In [114]:
# training
for k in range(10):
    # forward pass
    ypred = [n(x) for x in xs]
    loss = sum((yout - ygt)**2 for ygt, yout in zip(ys,ypred))

    # reset grads to zero before backward() to avoid adding previous grad to new one
    for p in n.parameters():
        p.grad = 0.0


    # backward pass
    loss.backward()

    # update
    for p in n.parameters():
        p.data += -0.05 * p.grad

    print(f'iteraion: {k},  loss : {loss.data}') 


iteraion: 0,  loss : 0.011110739880731118
iteraion: 1,  loss : 0.010801051252372084
iteraion: 2,  loss : 0.010504783988737072
iteraion: 3,  loss : 0.010221389613457796
iteraion: 4,  loss : 0.009950343458716933
iteraion: 5,  loss : 0.009691139158855647
iteraion: 6,  loss : 0.009443284584515186
iteraion: 7,  loss : 0.009206299073125477
iteraion: 8,  loss : 0.008979711777622404
iteraion: 9,  loss : 0.008763060934616741


In [115]:
ypred

[Value(data=0.9441546479664961),
 Value(data=-0.9657064646880728),
 Value(data=-0.9515482030979876),
 Value(data=0.9539485678436979)]